Question 17: Daily Logins
Difficulty: Easy
Link: https://www.interviewquery.com/questions/daily-logins?playlist=14-days-of-pandas

Problem Description:
===================
Let’s say we are given a table `user_logins`.

Using this table, calculate how many users logged in an identical number of times on January 1st,
2022.

For example, it may be the case that on that date three users logged in seven times, four users
logged in five times, two users logged in ten times, etc.

**Example:**

**Input:**

`user_logins` table

Column | Type  
---|---  
`id` | INTEGER  
`user_id` | INTEGER  
`login_date` | DATETIME  
  
**Output:**

Column | Type  
---|---  
`number_of_logins` | INTEGER  
`number_of_users` | INTEGER


In [94]:
# Question 17: Daily Logins
# 
# Let’s say we are given a table user_logins.
#
# Using this table, calculate how many users logged in an identical number of times on
# January 1st,
# 2022.
#
# For example, it may be the case that on that date three users logged in seven times,
# four users
# logged in five times, two users logged in ten times, etc.

import pandas as pd
import numpy as np
from datetime import datetime

# user_logins table mock data
user_logins_data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'user_id': [101, 101, 102, 103, 101, 102, 104, 105, 106, 107, 104, 108],
    'login_date': pd.to_datetime([
        '2022-01-01 08:00:00', # 101
        '2022-01-01 12:00:00', # 101
        '2022-01-01 09:30:00', # 102
        '2022-01-01 11:15:00', # 103
        '2022-01-01 18:45:00', # 101 (User 101 has 3 logins)
        '2022-01-01 20:00:00', # 102 (User 102 has 2 logins)
        '2022-01-01 10:00:00', # 104
        '2022-01-01 13:00:00', # 105
        '2022-01-01 15:30:00', # 106
        '2022-01-02 09:00:00', # 107 (Wrong Date - should be filtered)
        '2022-01-01 19:15:00', # 104 (User 104 has 2 logins)
        '2022-01-01 07:30:00', # 108 (User 103, 105, 106, 108 have 1 login)
    ]),
}
user_logins = pd.DataFrame(user_logins_data)

def solution():
    df = user_logins
    df['login_date'] = df['login_date'].dt.date.astype(str)
    df = user_logins[user_logins['login_date'] == '2022-01-01']
    df = df.groupby('user_id')['id'].count().reset_index().rename(columns = {'id':'login_count'})
    df = df.groupby('login_count')['user_id'].count().reset_index()

    return df
     

solution()

,login_count,user_id
0,1,4
1,2,2
2,3,1


### Hard Challenge 1: Maximum Gap Between Logins
For users who have logged in more than once, calculate the longest gap (in hours) between any two consecutive logins.

*Hint: You will need to rank/sort logins chronologically per user and use `.diff()` or `.shift()`.*

**Expected Columns:** `user_id`, `max_gap_hours`


In [17]:
def hard_1():
    df = user_logins.copy()
    df = df.sort_values(['user_id','login_date'])
    # Your code here
    df['gap_hrs'] = df.groupby('user_id')['login_date'].diff().dt.total_seconds()/3600
    df = df.groupby('user_id')['gap_hrs'].max().reset_index()
    return df

    
hard_1()

,user_id,gap_hrs
0,101,6.75
1,102,10.50
2,103,NaN
3,104,9.25
4,105,NaN
5,106,NaN
6,107,NaN
7,108,NaN


### Hard Challenge 2: Peak Concurrent Active Users (Session Overlap)
Assume every login represents an active session that lasts exactly **2 hours** from the `login_date`. Calculate the maximum number of concurrent active users at any given single moment.

*Hint: This is a classic overlaps problem. Think about transforming the data into chronological `login` (+1) and `logout` (-1) boundary events, and calculating a running sum (`.cumsum()`).*

**Expected Output:** A single integer representing the peak concurrent users.


In [21]:
def hard_2():
    df = user_logins.copy()
    df['logout_date'] = df['login_date'] + pd.Timedelta(hours=2)
    # Your code here
    df = df[['user_id','login_date','logout_date']]
    return df , type(df)

hard_2()

(    user_id          login_date         logout_date
 0       101 2022-01-01 08:00:00 2022-01-01 10:00:00
 1       101 2022-01-01 12:00:00 2022-01-01 14:00:00
 2       102 2022-01-01 09:30:00 2022-01-01 11:30:00
 3       103 2022-01-01 11:15:00 2022-01-01 13:15:00
 4       101 2022-01-01 18:45:00 2022-01-01 20:45:00
 5       102 2022-01-01 20:00:00 2022-01-01 22:00:00
 6       104 2022-01-01 10:00:00 2022-01-01 12:00:00
 7       105 2022-01-01 13:00:00 2022-01-01 15:00:00
 8       106 2022-01-01 15:30:00 2022-01-01 17:30:00
 9       107 2022-01-02 09:00:00 2022-01-02 11:00:00
 10      104 2022-01-01 19:15:00 2022-01-01 21:15:00
 11      108 2022-01-01 07:30:00 2022-01-01 09:30:00,
 pandas.core.frame.DataFrame)

### Hard Challenge 3: Login Shifts & Window Functions
Categorize each login into a "Shift" based on the time of day:
- `Morning`: 06:00 to 11:59
- `Afternoon`: 12:00 to 17:59
- `Evening`: 18:00 to 23:59

Then, write a query to find out which user was the **very first person** to log in during each shift on `2022-01-01`.

*Hint: You will need `.dt.hour`, and an equivalent to SQL's `row_number()` or `.idxmin()`, combined with grouping.*

**Expected Columns:** `shift`, `user_id`, `login_date`


In [ ]:
def hard_3():
    df = user_logins.copy()
    # Your code here
    pass

hard_3()

### Hard Challenge 4: Total Active Duration per User
Calculate the **total active duration** (in hours) for each user on `2022-01-01`. Active duration is defined as the time difference between their very first login and their very last login on that specific date. 

*Exclude users who only logged in once* (since their active duration would technically be 0).

*Hint: Group by `user_id`, aggregate to find the minimum and maximum `login_date`. Subtract the min from the max to get a timedelta, and then convert that timedelta into hours using `.dt.total_seconds() / 3600`.*

**Expected Columns:** `user_id`, `active_duration_hours`


In [ ]:
def hard_4():
    df = user_logins.copy()
    # Your code here
    pass

hard_4()